<a href="https://colab.research.google.com/github/y123ash/KNOVOS_Sentiment_Analysis/blob/main/Sentiment_Analysis_Sep9_2024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch.utils.data import TensorDataset

import pickle

from google.colab import drive

import tf_keras as tfk
import gensim.downloader as api
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
from tf_keras import models as md
from tf_keras import layers as ly
from tf_keras import losses as ls
from tf_keras import optimizers as oz
from tf_keras import metrics as ms
import numpy as np
import tensorflow as tf
#import drive
#import files
#from datasets import Dataset
from transformers import (BertTokenizerFast, TFBertTokenizer, BertTokenizer, RobertaTokenizerFast, DataCollatorWithPadding, TFRobertaForSequenceClassification, TFBertForSequenceClassification,TFBertModel, create_optimizer)
import torch, pathlib, os, re, string, time
from torch import tensor
import pandas as pd
import transformers as trf

from tf_keras.src.optimizers.schedules.learning_rate_schedule import CosineDecay

import accelerate
from tf_keras.src.losses import kl_divergence
from transformers import Trainer, TrainingArguments
from tf_keras.src.optimizers.adam import Adam


In [ ]:
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:

BATCH_SIZE=8
dataset2=pd.read_parquet(r"/content/drive/MyDrive/default_train_0000.parquet", engine="pyarrow")
dataset3=pd.read_parquet(r"/content/drive/MyDrive/default_validation_0000.parquet", engine="pyarrow")
cdataset2=dataset2
dataset2_text=dataset2["text"].tolist()
dataset3_text=dataset3["text"].tolist()
cdataset3=dataset3
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
tokenizer = BertTokenizerFast.from_pretrained(model_max_length=1024, pretrained_model_name_or_path='bert-base-uncased')
config = AutoConfig.from_pretrained(pretrained_model_name_or_path='bert-base-uncased')
tdataset2_text = tokenizer(dataset2_text, padding=True, truncation=True, max_length=128)
tdataset3_text = tokenizer(dataset3_text, padding=True, truncation=True, max_length=128)
tdataset2_dict=tdataset2_text.data
tdataset3_dict=tdataset3_text.data
cTdataset2 = pd.DataFrame.from_dict(tdataset2_dict)

cdataset2 = pd.concat([cdataset2, cTdataset2], axis=1)
tokenized_dataset2 = cdataset2

cTdataset3 = pd.DataFrame.from_dict(tdataset3_dict)

cdataset3 = pd.concat([cdataset3, cTdataset3], axis=1)
tokenized_dataset3 = cdataset3

numeric_feature_names2 = ['input_ids','token_type_ids', 'attention_mask','label']
numeric_features2 = tokenized_dataset2[numeric_feature_names2]

numeric_feature_names3 = ['input_ids','token_type_ids', 'attention_mask','label']
numeric_features3 = tokenized_dataset3[numeric_feature_names3]




from torch.utils.data import Dataset, DataLoader
class NamedTensorDataset(Dataset):
    def __init__(self, tensor, field_names, batch_size):
        self.tensor = tensor
        self.field_names = [f'feature_{i}' for i in range(tensor.shape[1])]
        self.batch_size = batch_size

    def __getitem__(self, index):
        batch_index = index * self.batch_size
        batch_tensor = self.tensor[batch_index:batch_index + self.batch_size]
        return {name: batch_tensor[:, i] for i, name in enumerate(self.field_names)}

    def __len__(self):
        return (self.tensor.shape[0] + self.batch_size - 1) // self.batch_size

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/default_train_0000.parquet'

In [ ]:
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import tensorflow as tf
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoConfig, Trainer, TrainingArguments)
# Load the data
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
dataset2 = pd.read_parquet("/content/drive/MyDrive/default_train_0000.parquet", engine="pyarrow")
dataset3 = pd.read_parquet("/content/drive/MyDrive/default_validation_0000.parquet", engine="pyarrow")

# Tokenize the data
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tdataset2_text = tokenizer(dataset2["text"].tolist(), padding=True, truncation=True, max_length=128)
tdataset3_text = tokenizer(dataset3["text"].tolist(), padding=True, truncation=True, max_length=128)

# Create PyTorch tensors
labels2 = torch.tensor(dataset2['label'].values)
labels3 = torch.tensor(dataset3['label'].values)

input_ids2 = torch.tensor(tdataset2_text['input_ids'])
token_type_ids2 = torch.tensor(tdataset2_text['token_type_ids'])
attention_mask2 = torch.tensor(tdataset2_text['attention_mask'])

input_ids3 = torch.tensor(tdataset3_text['input_ids'])
token_type_ids3 = torch.tensor(tdataset3_text['token_type_ids'])
attention_mask3 = torch.tensor(tdataset3_text['attention_mask'])

"""
class JoyoMeterDataset(Dataset):

  def __init__(self, features, target):
    self.features = features
    self.target = target

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
     feature = self.features[index]
     target = self.target[index]

     return feature, target
"""

from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __init__(self, input_ids, token_type_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.token_type_ids = token_type_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'token_type_ids': self.token_type_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }
"""

train_dataset = JoyoMeterDataset(tweets_train, y_train)
val_dataset = JoyoMeterDataset(tweets_val, y_val)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=True)
"""

# Create dataset instances
tensor_dataset2 = MyDataset(input_ids2, token_type_ids2, attention_mask2, labels2)
tensor_dataset3 = MyDataset(input_ids3, token_type_ids3, attention_mask3, labels3)

# Create PyTorch datasets
#tensor_dataset2 = TensorDataset(input_ids2, token_type_ids2, attention_mask2, labels2)
#tensor_dataset3 = TensorDataset(input_ids3, token_type_ids3, attention_mask3, labels3)

# Create DataLoaders
BATCH_SIZE = 8
data_loader2 = DataLoader(tensor_dataset2, batch_size=BATCH_SIZE, shuffle=True)
data_loader3 = DataLoader(tensor_dataset3, batch_size=BATCH_SIZE, shuffle=False)
"""
class BertRegression(nn.Module):

    def __init__(self, model_name):

        super(BertRegression, self).__init__()

        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = torch.nn.Dropout(0.2) #Regularization
        #Add a linear layer for regression
        self.regressor = torch.nn.Linear(self.bert.config.hidden_size, 1)


    def forward(self, input_ids, attention_mask):

        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output

        dropout_output = self.dropout(pooled_output)

        linear_output = self.regressor(pooled_output)

        return linear_output
"""
# Define the custom Trainer class
class ImageDistilTrainer(Trainer):
    def __init__(self, student_model=None, teacher_model=None, temperature=1.0, *args, **kwargs):
        super().__init__(model=student_model, *args, **kwargs)
        self.teacher = teacher_model
        self.temperature = temperature

    def compute_loss(self, model, inputs, return_outputs=False):
        student_outputs = model(**inputs)
        with torch.no_grad():
            teacher_outputs = self.teacher(**inputs)

        # Calculate distillation loss using logits or predictions
        student_logits = student_outputs.logits / self.temperature
        teacher_logits = teacher_outputs.logits / self.temperature

        # Use predictions for distillation loss if num_labels is 1
        if student_outputs.logits.shape[1] == 1:
            loss = F.kl_div(
                F.logsigmoid(student_logits.squeeze()),
                torch.sigmoid(teacher_logits.squeeze()),
                reduction="batchmean"
            )
            # For binary classification, use BCEWithLogitsLoss
            loss += F.binary_cross_entropy_with_logits(
                student_logits.squeeze(),
                inputs['labels'].float()
            )
        else:
            loss = F.kl_div(F.log_softmax(student_logits, dim=-1),
                            F.softmax(teacher_logits, dim=-1),
                            reduction="batchmean")
            # For multi-class classification, use cross_entropy
            loss += F.cross_entropy(student_logits, inputs['labels'])

        return (loss, student_outputs) if return_outputs else loss

# Load the models
config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=1)
student_model = AutoModelForSequenceClassification.from_config(config)
teacher_model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', config=config)

# Freeze teacher model
for param in teacher_model.parameters():
    param.requires_grad = False

"""
# Define training arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/output",
    num_train_epochs=3,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    bert_regression_model = BertRegression(model_name=checkpoint).to(device)
    bert_regression_model.train()
"""
checkpoint = 'bert-base-uncased'

bert_regression_model =  AutoModelForSequenceClassification.from_pretrained(model_name=checkpoint)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

optim = torch.optim.AdamW(bert_regression_model.parameters(), lr=1e-5)
criterion = torch.nn.MSELoss()

epochs = 4

train_losses = []
val_losses = []

loss = None

for epoch in range(epochs):
  bert_regression_model.train()
  train_loss = 0
  val_loss = 0

  train_len = len(data_loader2)
  val_len = len(data_loader3)

  for tweets, score in train_dataloader:

    input = tokenizer.batch_encode_plus(tweets, return_tensors='pt', add_special_tokens=True, padding=True).to(device)

    output = bert_regression_model(input_ids=input.input_ids, attention_mask=input.attention_mask)

    loss = criterion(output.squeeze(), score)
    loss.backward()

    optim.step()
    optim.zero_grad()

    train_loss += loss.item()

  train_loss_avg = train_loss / train_len
  train_losses.append(train_loss_avg)

  with torch.no_grad():

    for tweets, score in val_dataloader:

      input = tokenizer.batch_encode_plus(tweets, add_special_tokens=True, padding=True, return_tensors='pt').to(device)
      output = bert_regression_model(input_ids=input.input_ids, attention_mask=input.attention_mask)

      loss = criterion(output.squeeze(), score)
      val_loss += loss.item()

    avg_val_loss = val_loss / val_len
    val_losses.append(avg_val_loss)

  print("Epoch {}/{}, Train Loss {:0.3f} Val Loss {:0.3f}".format(epoch+1, epochs, train_loss_avg, avg_val_loss))

In [ ]:
import numpy as np
from scipy.special import softmax
import csv
import urllib.request
#eval_results = model.evaluate(tf_val_dataset)
#print(f"Test accuracy: {eval_results['eval_accuracy']:.4f}")

input_text = input('Enter a sentence: ')
input_ids = tokenizer.encode_plus(input_text,
                                   add_special_tokens=True,
                                   max_length=max_length,
                                   return_attention_mask=True,
                                   return_tensors='tf',
                                   padding='max_length',
                                   truncation=True)

output = model(input_ids['input_ids'], attention_mask=input_ids['attention_mask'])
print(output.logits)
scores = output[0][0].detach().numpy()
scores = softmax(scores)
#ranking = np.argsort(scores)
#ranking = ranking[::-1]
labels={ 0: "positive", 1: "neutral", 2: "negative"}
ranking = np.argsort(scores)
ranking = ranking[::-1]
for i in range(scores.shape[0]):
    l = labels[ranking[i]]
    s = scores[ranking[i]]
    print(f"{i + 1}) {l} {np.round(float(s), 4)}")

In [ ]:
h=8

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig

# Load the data
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
dataset2 = pd.read_parquet("/content/drive/MyDrive/default_train_0000.parquet", engine="pyarrow")
dataset3 = pd.read_parquet("/content/drive/MyDrive/default_validation_0000.parquet", engine="pyarrow")

# Tokenize the data
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tdataset2_text = tokenizer(dataset2["text"].tolist(), padding=True, truncation=True, max_length=128, return_tensors='pt')
tdataset3_text = tokenizer(dataset3["text"].tolist(), padding=True, truncation=True, max_length=128, return_tensors='pt')

class MyDataset(Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }

# Create dataset instances
tensor_dataset2 = MyDataset(tdataset2_text['input_ids'], tdataset2_text['attention_mask'], torch.tensor(dataset2['label'].values))
tensor_dataset3 = MyDataset(tdataset3_text['input_ids'], tdataset3_text['attention_mask'], torch.tensor(dataset3['label'].values))

# Create DataLoaders
BATCH_SIZE = 8
data_loader2 = DataLoader(tensor_dataset2, batch_size=BATCH_SIZE, shuffle=True)
data_loader3 = DataLoader(tensor_dataset3, batch_size=BATCH_SIZE, shuffle=False)

# Define the model
# ...

# Define the model
labels = torch.tensor(dataset2['label'].values)
num_classes = torch.unique(labels).size(0)
config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=num_classes)
model = AutoModelForSequenceClassification.from_config(config)

# ...
#config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=torch.unique(labels).size(0))
#model = AutoModelForSequenceClassification.from_config(config)


# Define the optimizer and loss function
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optim = torch.optim.AdamW(model.parameters(), lr=1e-5)
criterion = torch.nn.CrossEntropyLoss()

# Train the model
for epoch in range(4):
    model.train()
    total_loss = 0
    for batch in data_loader2:
        labels = batch['labels'].to(device)
        assert torch.all((labels >= 0) & (labels < num_classes)), "Labels out of bounds"
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optim.zero_grad()

        outputs = model(input_ids, attention_mask=attention_mask)
        #loss = criterion(outputs.logits, labels)
        # import torch
        #import torch.nn as nn

        # Assume 'outputs' is the output of your model, and 'labels' is the tensor of labels
        #num_classes = 3  # Specify the number of classes
        #criterion = nn.CrossEntropyLoss(num_classes=num_classes)
        criterion = nn.CrossEntropyLoss()
        loss = criterion(outputs.logits, labels)
        loss.backward()
        optim.step()

        total_loss += loss.item()

    print(f'Epoch {epoch+1}, Loss: {total_loss / len(data_loader2)}')

    model.eval()
    total_correct = 0
    with torch.no_grad():
        for batch in data_loader3:
            labels = batch['labels'].to(device)
            assert torch.all((labels >= 0) & (labels < num_classes)), "Labels out of bounds"
            # ...
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            _, predicted = torch.max(outputs.logits, 1)
            total_correct += (predicted == labels).sum().item()

    accuracy = total_correct / len(dataset3)
    print(f'Epoch {epoch+1}, Val Accuracy: {accuracy:.4f}')

ValueError: mount failed

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig

# Load the data
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
dataset2 = pd.read_parquet("/content/drive/MyDrive/default_train_0000.parquet", engine="pyarrow")
dataset3 = pd.read_parquet("/content/drive/MyDrive/default_validation_0000.parquet", engine="pyarrow")

# Tokenize the data
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tdataset2_text = tokenizer(dataset2["text"].tolist(), padding=True, truncation=True, max_length=128, return_tensors='pt')
tdataset3_text = tokenizer(dataset3["text"].tolist(), padding=True, truncation=True, max_length=128, return_tensors='pt')

class MyDataset(Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }

# Create dataset instances
tensor_dataset2 = MyDataset(tdataset2_text['input_ids'], tdataset2_text['attention_mask'], torch.tensor(dataset2['label'].values))
tensor_dataset3 = MyDataset(tdataset3_text['input_ids'], tdataset3_text['attention_mask'], torch.tensor(dataset3['label'].values))

# Create DataLoaders
BATCH_SIZE = 8
data_loader2 = DataLoader(tensor_dataset2, batch_size=BATCH_SIZE, shuffle=True)
data_loader3 = DataLoader(tensor_dataset3, batch_size=BATCH_SIZE, shuffle=False)

# Define the model
# ...

# Define the model
labels = torch.tensor(dataset2['label'].values)
num_classes = torch.unique(labels).size(0)
config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=num_classes)
model = AutoModelForSequenceClassification.from_config(config)

# ...
#config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=torch.unique(labels).size(0))
#model = AutoModelForSequenceClassification.from_config(config)


# Define the optimizer and loss function
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optim = torch.optim.AdamW(model.parameters(), lr=1e-5)
criterion = torch.nn.CrossEntropyLoss()

# Train the model
for epoch in range(4):
    model.train()
    total_loss = 0
    for batch in data_loader2:
        labels = batch['labels'].to(device)
        assert torch.all((labels >= 0) & (labels < num_classes)), "Labels out of bounds"
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optim.zero_grad()

        outputs = model(input_ids, attention_mask=attention_mask)
        #loss = criterion(outputs.logits, labels)
        # import torch
        #import torch.nn as nn

        # Assume 'outputs' is the output of your model, and 'labels' is the tensor of labels
        #num_classes = 3  # Specify the number of classes
        #criterion = nn.CrossEntropyLoss(num_classes=num_classes)
        criterion = nn.CrossEntropyLoss()
        loss = criterion(outputs.logits, labels)
        loss.backward()
        optim.step()

        total_loss += loss.item()

    print(f'Epoch {epoch+1}, Loss: {total_loss / len(data_loader2)}')

    model.eval()
    total_correct = 0
    with torch.no_grad():
        for batch in data_loader3:
            labels = batch['labels'].to(device)
            assert torch.all((labels >= 0) & (labels < num_classes)), "Labels out of bounds"
            # ...
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            _, predicted = torch.max(outputs.logits, 1)
            total_correct += (predicted == labels).sum().item()

    accuracy = total_correct / len(dataset3)
    print(f'Epoch {epoch+1}, Val Accuracy: {accuracy:.4f}')

Mounted at /content/drive


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Epoch 1, Loss: 0.8725104388677065
Epoch 1, Val Accuracy: 0.6771
Epoch 2, Loss: 0.6769422010549057
Epoch 2, Val Accuracy: 0.7303


In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig

# Load the data
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
dataset2 = pd.read_parquet("/content/drive/MyDrive/default_train_0000.parquet", engine="pyarrow")
dataset3 = pd.read_parquet("/content/drive/MyDrive/default_validation_0000.parquet", engine="pyarrow")

# Tokenize the data
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tdataset2_text = tokenizer(dataset2["text"].tolist(), padding=True, truncation=True, max_length=128, return_tensors='pt')
tdataset3_text = tokenizer(dataset3["text"].tolist(), padding=True, truncation=True, max_length=128, return_tensors='pt')

class MyDataset(Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }

# Create dataset instances
tensor_dataset2 = MyDataset(tdataset2_text['input_ids'], tdataset2_text['attention_mask'], torch.tensor(dataset2['label'].values))
tensor_dataset3 = MyDataset(tdataset3_text['input_ids'], tdataset3_text['attention_mask'], torch.tensor(dataset3['label'].values))

# Create DataLoaders
BATCH_SIZE = 8
data_loader2 = DataLoader(tensor_dataset2, batch_size=BATCH_SIZE, shuffle=True)
data_loader3 = DataLoader(tensor_dataset3, batch_size=BATCH_SIZE, shuffle=False)

# Define the model
# ...

# Define the model
labels = torch.tensor(dataset2['label'].values)
num_classes = torch.unique(labels).size(0)
config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=num_classes)
model = AutoModelForSequenceClassification.from_config(config)

# ...
#config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=torch.unique(labels).size(0))
#model = AutoModelForSequenceClassification.from_config(config)


# Define the optimizer and loss function
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optim = torch.optim.AdamW(model.parameters(), lr=1e-5)
criterion = torch.nn.CrossEntropyLoss()

# Train the model
for epoch in range(4):
    model.train()
    total_loss = 0
    for batch in data_loader2:
        labels = batch['labels'].to(device)
        assert torch.all((labels >= 0) & (labels < num_classes)), "Labels out of bounds"
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optim.zero_grad()

        outputs = model(input_ids, attention_mask=attention_mask)
        #loss = criterion(outputs.logits, labels)
        # import torch
        #import torch.nn as nn

        # Assume 'outputs' is the output of your model, and 'labels' is the tensor of labels
        #num_classes = 3  # Specify the number of classes
        #criterion = nn.CrossEntropyLoss(num_classes=num_classes)
        criterion = nn.CrossEntropyLoss()
        loss = criterion(outputs.logits, labels)
        loss.backward()
        optim.step()

        total_loss += loss.item()

    print(f'Epoch {epoch+1}, Loss: {total_loss / len(data_loader2)}')

    model.eval()
    total_correct = 0
    with torch.no_grad():
        for batch in data_loader3:
            labels = batch['labels'].to(device)
            assert torch.all((labels >= 0) & (labels < num_classes)), "Labels out of bounds"
            # ...
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            _, predicted = torch.max(outputs.logits, 1)
            total_correct += (predicted == labels).sum().item()

    accuracy = total_correct / len(dataset3)
    print(f'Epoch {epoch+1}, Val Accuracy: {accuracy:.4f}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


IndexError: Target 2 is out of bounds.

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig

# Load the data
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
dataset2 = pd.read_parquet("/content/drive/MyDrive/default_train_0000.parquet", engine="pyarrow")
dataset3 = pd.read_parquet("/content/drive/MyDrive/default_validation_0000.parquet", engine="pyarrow")

# Tokenize the data
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tdataset2_text = tokenizer(dataset2["text"].tolist(), padding=True, truncation=True, max_length=128, return_tensors='pt')
tdataset3_text = tokenizer(dataset3["text"].tolist(), padding=True, truncation=True, max_length=128, return_tensors='pt')

class MyDataset(Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }

# Create dataset instances
tensor_dataset2 = MyDataset(tdataset2_text['input_ids'], tdataset2_text['attention_mask'], torch.tensor(dataset2['label'].values))
tensor_dataset3 = MyDataset(tdataset3_text['input_ids'], tdataset3_text['attention_mask'], torch.tensor(dataset3['label'].values))

# Create DataLoaders
BATCH_SIZE = 8
data_loader2 = DataLoader(tensor_dataset2, batch_size=BATCH_SIZE, shuffle=True)
data_loader3 = DataLoader(tensor_dataset3, batch_size=BATCH_SIZE, shuffle=False)

# Define the model

config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=torch.unique(labels).size(0))
model = AutoModelForSequenceClassification.from_config(config)


# Define the optimizer and loss function
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optim = torch.optim.AdamW(model.parameters(), lr=1e-5)
criterion = torch.nn.CrossEntropyLoss()

# Train the model
for epoch in range(4):
    model.train()
    total_loss = 0
    for batch in data_loader2:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optim.zero_grad()

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        #loss = criterion(outputs.logits, labels)
        # import torch
        #import torch.nn as nn

        # Assume 'outputs' is the output of your model, and 'labels' is the tensor of labels
        #num_classes = 3  # Specify the number of classes
        #criterion = nn.CrossEntropyLoss(num_classes=num_classes)
        criterion = nn.CrossEntropyLoss()
        loss = criterion(outputs, labels)
        loss.backward()
        optim.step()

        total_loss += loss.item()

    print(f'Epoch {epoch+1}, Loss: {total_loss / len(data_loader2)}')

    model.eval()
    total_correct = 0
    with torch.no_grad():
        for batch in data_loader3:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            _, predicted = torch.max(outputs.logits, 1)
            total_correct += (predicted == labels).sum().item()

    accuracy = total_correct / len(dataset3)
    print(f'Epoch {epoch+1}, Val Accuracy: {accuracy:.4f}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


TypeError: cross_entropy_loss(): argument 'input' (position 1) must be Tensor, not SequenceClassifierOutput

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig

# Load the data
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
dataset2 = pd.read_parquet("/content/drive/MyDrive/default_train_0000.parquet", engine="pyarrow")
dataset3 = pd.read_parquet("/content/drive/MyDrive/default_validation_0000.parquet", engine="pyarrow")

# Tokenize the data
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tdataset2_text = tokenizer(dataset2["text"].tolist(), padding=True, truncation=True, max_length=128, return_tensors='pt')
tdataset3_text = tokenizer(dataset3["text"].tolist(), padding=True, truncation=True, max_length=128, return_tensors='pt')

class MyDataset(Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }

# Create dataset instances
tensor_dataset2 = MyDataset(tdataset2_text['input_ids'], tdataset2_text['attention_mask'], torch.tensor(dataset2['label'].values))
tensor_dataset3 = MyDataset(tdataset3_text['input_ids'], tdataset3_text['attention_mask'], torch.tensor(dataset3['label'].values))

# Create DataLoaders
BATCH_SIZE = 8
data_loader2 = DataLoader(tensor_dataset2, batch_size=BATCH_SIZE, shuffle=True)
data_loader3 = DataLoader(tensor_dataset3, batch_size=BATCH_SIZE, shuffle=False)

# Define the model
config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=1)
model = AutoModelForSequenceClassification.from_config(config)

# Define the optimizer and loss function
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optim = torch.optim.AdamW(model.parameters(), lr=1e-5)
criterion = torch.nn.CrossEntropyLoss()

# Train the model
for epoch in range(4):
    model.train()
    total_loss = 0
    for batch in data_loader2:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optim.zero_grad()

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        #loss = criterion(outputs.logits, labels)
        # import torch
        #import torch.nn as nn

        # Assume 'outputs' is the output of your model, and 'labels' is the tensor of labels
        num_classes = 3  # Specify the number of classes
        criterion = nn.CrossEntropyLoss(num_classes=num_classes)

        loss = criterion(outputs, labels)
        loss.backward()
        optim.step()

        total_loss += loss.item()

    print(f'Epoch {epoch+1}, Loss: {total_loss / len(data_loader2)}')

    model.eval()
    total_correct = 0
    with torch.no_grad():
        for batch in data_loader3:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            _, predicted = torch.max(outputs.logits, 1)
            total_correct += (predicted == labels).sum().item()

    accuracy = total_correct / len(dataset3)
    print(f'Epoch {epoch+1}, Val Accuracy: {accuracy:.4f}')

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig

# Load the data
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
dataset2 = pd.read_parquet("/content/drive/MyDrive/default_train_0000.parquet", engine="pyarrow")
dataset3 = pd.read_parquet("/content/drive/MyDrive/default_validation_0000.parquet", engine="pyarrow")

# Tokenize the data
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tdataset2_text = tokenizer(dataset2["text"].tolist(), padding=True, truncation=True, max_length=128, return_tensors='pt')
tdataset3_text = tokenizer(dataset3["text"].tolist(), padding=True, truncation=True, max_length=128, return_tensors='pt')

class MyDataset(Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }

# Create dataset instances
tensor_dataset2 = MyDataset(tdataset2_text['input_ids'], tdataset2_text['attention_mask'], torch.tensor(dataset2['label'].values))
tensor_dataset3 = MyDataset(tdataset3_text['input_ids'], tdataset3_text['attention_mask'], torch.tensor(dataset3['label'].values))

# Create DataLoaders
BATCH_SIZE = 8
data_loader2 = DataLoader(tensor_dataset2, batch_size=BATCH_SIZE, shuffle=True)
data_loader3 = DataLoader(tensor_dataset3, batch_size=BATCH_SIZE, shuffle=False)

# Define the model
config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=1)
model = AutoModelForSequenceClassification.from_config(config)

# Define the optimizer and loss function
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optim = torch.optim.AdamW(model.parameters(), lr=1e-5)
criterion = torch.nn.CrossEntropyLoss()

# Train the model
for epoch in range(4):
    model.train()
    total_loss = 0
    for batch in data_loader2:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optim.zero_grad()

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        #loss = criterion(outputs.logits, labels)
        # import torch
        #import torch.nn as nn

        # Assume 'outputs' is the output of your model, and 'labels' is the tensor of labels
        num_classes = 3  # Specify the number of classes
        criterion = nn.CrossEntropyLoss(num_classes=num_classes)

        loss = criterion(outputs, labels)
        loss.backward()
        optim.step()

        total_loss += loss.item()

    print(f'Epoch {epoch+1}, Loss: {total_loss / len(data_loader2)}')

    model.eval()
    total_correct = 0
    with torch.no_grad():
        for batch in data_loader3:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            _, predicted = torch.max(outputs.logits, 1)
            total_correct += (predicted == labels).sum().item()

    accuracy = total_correct / len(dataset3)
    print(f'Epoch {epoch+1}, Val Accuracy: {accuracy:.4f}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


TypeError: CrossEntropyLoss.__init__() got an unexpected keyword argument 'num_classes'

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig

# Load the data
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
dataset2 = pd.read_parquet("/content/drive/MyDrive/default_train_0000.parquet", engine="pyarrow")
dataset3 = pd.read_parquet("/content/drive/MyDrive/default_validation_0000.parquet", engine="pyarrow")

# Tokenize the data
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tdataset2_text = tokenizer(dataset2["text"].tolist(), padding=True, truncation=True, max_length=128, return_tensors='pt')
tdataset3_text = tokenizer(dataset3["text"].tolist(), padding=True, truncation=True, max_length=128, return_tensors='pt')

class MyDataset(Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }

# Create dataset instances
tensor_dataset2 = MyDataset(tdataset2_text['input_ids'], tdataset2_text['attention_mask'], torch.tensor(dataset2['label'].values))
tensor_dataset3 = MyDataset(tdataset3_text['input_ids'], tdataset3_text['attention_mask'], torch.tensor(dataset3['label'].values))

# Create DataLoaders
BATCH_SIZE = 8
data_loader2 = DataLoader(tensor_dataset2, batch_size=BATCH_SIZE, shuffle=True)
data_loader3 = DataLoader(tensor_dataset3, batch_size=BATCH_SIZE, shuffle=False)

# Define the model
config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=1)
model = AutoModelForSequenceClassification.from_config(config)

# Define the optimizer and loss function
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optim = torch.optim.AdamW(model.parameters(), lr=1e-5)
criterion = torch.nn.CrossEntropyLoss()

# Train the model
for epoch in range(4):
    model.train()
    total_loss = 0
    for batch in data_loader2:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optim.zero_grad()

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        #loss = criterion(outputs.logits, labels)
        # import torch
        #import torch.nn as nn

        # Assume 'outputs' is the output of your model, and 'labels' is the tensor of labels
        num_classes = 3  # Specify the number of classes
        criterion = nn.CrossEntropyLoss(num_classes=num_classes)

        loss = criterion(outputs, labels)
        loss.backward()
        optim.step()

        total_loss += loss.item()

    print(f'Epoch {epoch+1}, Loss: {total_loss / len(data_loader2)}')

    model.eval()
    total_correct = 0
    with torch.no_grad():
        for batch in data_loader3:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            _, predicted = torch.max(outputs.logits, 1)
            total_correct += (predicted == labels).sum().item()

    accuracy = total_correct / len(dataset3)
    print(f'Epoch {epoch+1}, Val Accuracy: {accuracy:.4f}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


NameError: name 'nn' is not defined

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig

# Load the data
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
dataset2 = pd.read_parquet("/content/drive/MyDrive/default_train_0000.parquet", engine="pyarrow")
dataset3 = pd.read_parquet("/content/drive/MyDrive/default_validation_0000.parquet", engine="pyarrow")

# Tokenize the data
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tdataset2_text = tokenizer(dataset2["text"].tolist(), padding=True, truncation=True, max_length=128, return_tensors='pt')
tdataset3_text = tokenizer(dataset3["text"].tolist(), padding=True, truncation=True, max_length=128, return_tensors='pt')

class MyDataset(Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }

# Create dataset instances
tensor_dataset2 = MyDataset(tdataset2_text['input_ids'], tdataset2_text['attention_mask'], torch.tensor(dataset2['label'].values))
tensor_dataset3 = MyDataset(tdataset3_text['input_ids'], tdataset3_text['attention_mask'], torch.tensor(dataset3['label'].values))

# Create DataLoaders
BATCH_SIZE = 8
data_loader2 = DataLoader(tensor_dataset2, batch_size=BATCH_SIZE, shuffle=True)
data_loader3 = DataLoader(tensor_dataset3, batch_size=BATCH_SIZE, shuffle=False)

# Define the model
config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=1)
model = AutoModelForSequenceClassification.from_config(config)

# Define the optimizer and loss function
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optim = torch.optim.AdamW(model.parameters(), lr=1e-5)
criterion = torch.nn.CrossEntropyLoss()

# Train the model
for epoch in range(4):
    model.train()
    total_loss = 0
    for batch in data_loader2:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optim.zero_grad()

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        #loss = criterion(outputs.logits, labels)
        # import torch
        #import torch.nn as nn

        # Assume 'outputs' is the output of your model, and 'labels' is the tensor of labels
        num_classes = 3  # Specify the number of classes
        criterion = nn.CrossEntropyLoss(num_classes=num_classes)

        loss = criterion(outputs, labels)
        loss.backward()
        optim.step()

        total_loss += loss.item()

    print(f'Epoch {epoch+1}, Loss: {total_loss / len(data_loader2)}')

    model.eval()
    total_correct = 0
    with torch.no_grad():
        for batch in data_loader3:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            _, predicted = torch.max(outputs.logits, 1)
            total_correct += (predicted == labels).sum().item()

    accuracy = total_correct / len(dataset3)
    print(f'Epoch {epoch+1}, Val Accuracy: {accuracy:.4f}')

Mounted at /content/drive


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

IndexError: Target 2 is out of bounds.